# cs246 - 01导论


前提:
- 有计算机系统的知识背景(数据存储)
- 有数据库知识背景(数据管理)

CS246 数据挖掘/大数据/预测分析/数据科学/机器学习 

数据挖掘的方法论:
- 发现数据中符合人类理解的模式 : 聚类
- 预测变量未来的值 : 推荐

cs246 结合了机器学习,统计学,人工智能,数据库, 重点关注数据规模,算法,计算架构,大规模数据的自动化处理

数据类型
- 高维数据
- 图数据
- 无限数据
- 带标签数据

计算架构:
- MapReduce (MR计算架构)
- Streams and online algorithms (流式架构和在线算法)
- Single machine in-memory (单机系统)

解决问题:
- 推荐系统
- 市场分析
- 虚假评价(spam detection)
- 文档重复检测

模型:
- 线性代数 (SVD)
- 优化方法(stochastic gradient descent)
- 动态编程 ( frequent itemsets )
- hashing (LSH,Bloom filters)
 
## 问题与方法,模型的结合:

- 高维数据: `局部敏感分析`,`聚类`,`降维`
- 图数据: `PageRank`,`SimRank`,`Network Analysis网路分析`,`Spam Dectection虚假检测`
- infinite data:`Filtering data streams数据流过滤`,`Web advertising广告`,`Queries on streams流式查询`
- machine learning: `SVM`,`Decision Trees决策树`,`Preceptron感知机` 
- apps: `推荐系统`,`关联规则association rules`,`文档重复检测`

相关课程:CS246H 介绍spark相关的技术与实现

## 大规模计算的挑战

- 如何分布计算?
- 如何便捷编写分布式程序?
- 如何提供容错,应对设备错误?

解决想法:
- 把计算送给数据
- 重复存储提供可靠性

`Spark/Hadoop`实现了解决方案
- 存储方案: HDFS
- 计算模型: MapReduce / Spark

## Distributed File System 分布式文件系统

Chunk servers(块存储服务器):
- 文件被分割成相邻的区块
- 每一个区块的大小一般在 16~64MB
- 每一个区块会冗余×2或者×2
- 尽量保证冗余块位于不同的机架中

Master Node
- `Name Node` in Hadoop's HDFS
- 主节点上存储了记录具体文件存储的`元文件(meta data)`
- 主节点的数也可以被备份

客户端接口要有简单的文件访问(权限):
- 要么和 master node 沟通, 以找到 chunk servers
- 要么直接从 chunk servers 获取数据

chunk会在集群间传播,如果有节点发生损坏,对于数据的获取可以无缝衔接.

> chunk server + master node 实现了数据存储的可靠性  
> 他们同时还作为计算节点,实现了将计算送给数据 


## 编程模型

### MapReduce

有很多种实现:
- Hadoop
- Spark
- Flink

### MapReduce编程环境

- 输入数据的分区
- 机器群之间的任务调度
- group by key 的实现 ( hadoop 中是 sort )
- 容错
- 管理机器之间的交流

### 错误处理

- map worker failure: 任务将重新调度(完成与否都), reduce workers将收到map task重新调度的通知
- reduce work failure: 只有执行中的任务会被重新调度



### **MapReduce的编程**模型的三步:

1. Map: 对于每一个元素应用用户编写的Map函数(map function).Mapper 会将 map function 应用在每一个输入元素上,一个计算单元(也是一个Map Task)中会包含很多Mapper.Map Function的输出是0,1或多个键值对
2. Group by key(**Sort and Group**): 排序和洗牌(shuffle), 系统根据 key-values对中的 key 对数据进行排序, 输出key-{values}的键值对
3. Reduce: 对 key - {values}应用用户编写的 Reduce 函数

> 问题: 是否需要所有的map函数执行结束才能执行reduce? Yes

MapReduce的应用:
- 分析网站数据,找出最热门的urls
- 统计机器翻译:统计每5个词在大文档库中出现的次数

文档词统计
```python
map(key, value)
# key: document name;value: text of document
  for each word w in value:
    emit(w,1)

reduce(key, values)
# key: a word; values: iterator over counts
  result = 0
  for each count v in values:
    result += v
  emit(key,result)
```

## Spark

spark主要解决了hadoop的两个局限:
- MR编程困难,很多任务很难通过map-reduce来描述
- 性能瓶颈,hadoop中 map-reduce 的任务会将结果写入disk

Spark是数据流系统,相比于:
- 运行任意数量的tasks/ranks
- 支持更多的函数不仅仅是Map/Reduce
- 通用的执行图 DAGs
- 兼容 hadoop
- RDD: chunk of data 的抽象
- 更加高级的API, ml, graphx, sql

### RDD的概念

- 存储记录,通常是键值对
- 集群间传播,只读
- 数据集缓存在内存,支持不同级别的存储,支持容错
- 可以从很多数据源创建
- 适用于需要对大规模数据元素执行相同操作的应用

Transformations:
- map,join,union,intersection,distinct
- 懒求值,在action之前不会执行任何运算

Actions:
- count
- collect
- reduce
- save

### Task Scheduler
- 支持通用的任务图
- Pipelines functions where possible
- Cache-aware data reuse & locality
- Partitioning-aware to avoid shuffles

### DataFrame & Dataset

- DataFrame: 数据按照 excel 的方式组织,关系型数据库 
- Dataset: DataFrame的拓展,支持类型安全,提供面向对象的编程接口

两者基于spark sql引擎,都可以变成RDD

## Spark Vs Hadoop MapReduce

- Spark通常更快
   - spark能够从内存直接获取数据,hadoop需要通过磁盘
   - 相比于MR,spark需要更多内存去执行更加丰富的操作
   - MR适用于简单任务,在简单任务上性能差异很小
- spark更加适合编程

算法花费的度量方式:
- communication cost = total I/O of all processes
- Elapsed communication cost = max I/O of all processes
- (Elapsed) computation cost = run time of processes